# 03 — Full-resolution inference and submission

**Purpose:** load a trained v2 checkpoint, inspect native-resolution predictions, and explicitly generate a structurally validated Kaggle CSV.  
**Inputs:** 180 local test JPEGs, trained v2 `best_model.pth`, selected threshold/area.  
**Outputs:** displayed diagnostics; a new `outputs/submission_v2_*.csv` only when `RUN_SUBMISSION=True`.  
**Dependencies:** project `.conda`, `src/filament_seg`, pycocotools.  
**Execution:** run top to bottom from root or `notebooks/`; full test generation defaults off. No training notebook state is required.  
**Runtime:** one-image preview can take seconds; 180-image inference can take minutes.  
**Related:** notebook 05 tunes parameters; this notebook creates the final file after review.

The local PQ implementation and structural CSV validator are diagnostic. Official scorer parity and Kaggle acceptance remain unverified.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'submission.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'submission.py').is_file(), cwd
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import load_config, get_device
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.inference import predict_tiled, iter_probability_instances
from filament_seg.evaluate import diagnostic_panoptic_quality, evaluate_full_resolution_dataset
from filament_seg.submission import encode_coco_counts, decode_coco_counts, validate_submission_rows, generate_submission

device = get_device()
print(f'Python: {sys.executable}')
print(f'Device: {device}')
assert Path(sys.executable).resolve() == (project_root / '.conda' / 'python.exe').resolve(), sys.executable


## 1. Load Trained Checkpoint

Load the best U-Net model (`best_model.pth`) without retraining.

In [ ]:
ckpt_path = project_root / 'checkpoints/unet_resnet34_v2/best_model.pth'
if not ckpt_path.is_file(): raise FileNotFoundError('Trained v2 checkpoint required: ' + str(ckpt_path))
model = build_model(pretrained=False).to(device)
state = load_checkpoint(ckpt_path, model, map_location=device)
assert state.get('epoch',0) > 0 and not any(h.get('smoke_test') for h in state.get('history',[]))
model.eval()
print('Trained checkpoint:', ckpt_path, 'epoch:', state.get('epoch'), 'best validation Dice:', state.get('best_val_dice'))

## 2. Full-Resolution Tiled Inference with Hann Window Blending

Demonstrate seamless 2048x2048 prediction with 512x512 tiles and 128px cosine overlap blending.

In [ ]:
test_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'test' / 'test_images'
test_images = sorted(test_dir.glob('*.jpeg'))
assert len(test_images) == 180, f'Expected 180 test images, found {len(test_images)}'

sample_test = test_images[0]
import cv2
img = cv2.imread(str(sample_test), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

prob_map = predict_tiled(model, img, tile_size=512, overlap=128, device=device)
print(f'Test image: {sample_test.name}')
print(f'Output shape: {prob_map.shape}, range: [{prob_map.min():.4f}, {prob_map.max():.4f}]')

## 3. Thresholding, Instance Extraction & Panoptic Quality (PQ)

Extract individual filament instances using historical baseline settings (threshold=0.75, min_area=300).

In [ ]:
train_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'train'
dataset = SolarFilamentDataset(train_dir / 'train_images', train_dir / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
indices = np.random.default_rng(42).permutation(len(dataset))
val_indices = indices[565:].tolist()

# Evaluate on sample validation image
val_sample = dataset[val_indices[0]]
val_prob = predict_tiled(model, val_sample['image'], tile_size=512, overlap=128, device=device)
gt_instances = [inst['mask'] for inst in dataset.iter_instances(val_indices[0], 0)]
pred_instances = list(iter_probability_instances(val_prob, threshold=0.75, min_area=300))

pq_res = diagnostic_panoptic_quality(gt_instances, pred_instances)
print(f'Sample Validation File: {val_sample["filename"]}')
print(f'Ground Truth Instances: {len(gt_instances)}, Predicted Instances: {len(pred_instances)}')
print(f'Diagnostic PQ: {pq_res["pq"]:.4f} (TP={pq_res["tp"]}, FP={pq_res["fp"]}, FN={pq_res["fn"]})')

## 4. Compressed COCO RLE Verification & Schema Conformance

Verify compressed COCO string counts round-trip encoding and structural schema validation.

In [ ]:
if pred_instances:
    sample_mask = pred_instances[0]
    counts = encode_coco_counts(sample_mask)
    decoded = decode_coco_counts(counts, sample_mask.shape)
    assert np.array_equal(decoded, sample_mask), 'RLE round trip failed!'
    print('COCO RLE Round-Trip: PASS')
    print(f'Encoded counts snippet: {counts[:40]}... (length: {len(counts)})')

## 5. Generate Submission

Generate competition submission CSV across test images using validated postprocessing parameters.

In [ ]:
RUN_SUBMISSION = False  # Explicit opt-in after reviewing model and post-processing settings.
SELECTED_THRESHOLD = 0.70  # historical v2 tuning reference; revalidate on holdout
SELECTED_MIN_AREA = 500
if RUN_SUBMISSION:
    from datetime import datetime, timezone
    output_path = project_root / 'outputs' / f'submission_v2_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.csv'
    df_submission = generate_submission(model, test_images, output_csv_path=output_path, device=device,
                                        threshold=SELECTED_THRESHOLD, min_area=SELECTED_MIN_AREA,
                                        tile_size=512, overlap=128)
    assert list(df_submission.columns) == ['filament_id','segmentation_rle']
    validate_submission_rows(df_submission.to_dict('records'), test_images)
    predicted_stems = {name.rsplit('_',1)[0] for name in df_submission['filament_id']}
    empty_images = {p.stem for p in test_images} - predicted_stems
    print('CSV:', output_path, 'images:', len(test_images), 'instances:', len(df_submission), 'empty predictions:', len(empty_images))
    print('Official Kaggle acceptance has not been tested.')
else:
    print('Submission generation skipped; existing CSV files are untouched.')